# Preprocessing

Writes `.xes.gz` logs from CSV sources, a 500-trace BPIC12 sample, and the CVS pharmacy zip.

Links to the source files are in the [repository README](../README.md#data).

Run from the repository root or from `analyses/`.


## Setup

In [ ]:
import gzip
import os
import zipfile
from pathlib import Path

import pandas as pd
from pm4py.objects.log.util import dataframe_utils
from pm4py.objects.conversion.log import converter as log_converter
from pm4py.objects.log.exporter.xes import exporter as xes_exporter

# Resolve project root whether cwd is the repo root or analyses/.
cwd = Path.cwd().resolve()
if (cwd / "data" / "real").is_dir():
    project_root = cwd
elif (cwd.parent / "data" / "real").is_dir():
    project_root = cwd.parent
else:
    raise FileNotFoundError(f"Could not locate data/real from cwd={cwd}")

def data_path(*parts: str) -> Path:
    return project_root.joinpath("data", *parts)

def xes_out(dataset_dir: str, stem: str) -> Path:
    """Build output path: data/real/<dataset_dir>/<stem>.xes.gz"""
    return data_path("real", dataset_dir, f"{stem}.xes.gz")

print("project_root =", project_root)


## Test log

Create a **500**-trace sample from BPIC 2012 to test the scripts (not to be used in formal evaluations).


In [ ]:
from pm4py.objects.log.importer.xes import importer as xes_importer
from pm4py.objects.log.obj import EventLog

log_path = data_path("real", "BPIC12", "BPI_Challenge_2012.xes.gz")
out_path = xes_out("TEST_BPIC12", "TEST_BPI_Challenge_2012")

log = xes_importer.apply(str(log_path))
number_traces = 500
sampled_log = EventLog(log[:number_traces])
os.makedirs(out_path.parent, exist_ok=True)
xes_exporter.apply(sampled_log, str(out_path))
print("Wrote", out_path, "with", number_traces, "traces")


## ACCRE-log

The log has been split into test and train. Each contains only completed traces. Hence, some traces that start in the train data and finish in the test data have been removed. Because we want to avoid detecting drifts due to these removed traces, we only include the train dataset.


In [ ]:
log_path = data_path("real", "ACCRE", "AcademicCredentials_train.csv.gz")
df = pd.read_csv(log_path, compression="gzip")

df["case:concept:name"] = df["case_id"]
df["concept:name"] = df["activity"]
df["time:timestamp"] = df["start_time"]
df["org:resource"] = df["resource"]

df = dataframe_utils.convert_timestamp_columns_in_df(df)
df = df.sort_values(by="time:timestamp").reset_index(drop=True)

event_log = log_converter.apply(df, variant=log_converter.Variants.TO_EVENT_LOG)
output_path = xes_out("ACCRE", "ACCRE")
xes_exporter.apply(event_log, str(output_path), parameters={"gzip": True})
print("Wrote", output_path, "| events:", len(df), "| traces:", df["case:concept:name"].nunique())


## BPIC14

In [ ]:
log_path = data_path("real", "BPIC14", "Detail_Incident_Activity.csv.gz")
df = pd.read_csv(log_path, delimiter=";", compression="gzip")

df["case:concept:name"] = df["Incident ID"]
df["concept:name"] = df["IncidentActivity_Type"]
df["time:timestamp"] = df["DateStamp"]

df = dataframe_utils.convert_timestamp_columns_in_df(df)
df = df.sort_values(by="time:timestamp").reset_index(drop=True)

event_log = log_converter.apply(df, variant=log_converter.Variants.TO_EVENT_LOG)
output_path = xes_out("BPIC14", "BPIC14")
xes_exporter.apply(event_log, str(output_path), parameters={"gzip": True})
print("Wrote", output_path, "| events:", len(df), "| traces:", df["case:concept:name"].nunique())


## CALL

The log has been split into test and train. Each contains only completed traces. Hence, some traces that start in the train data and finish in the test data have been removed. Because we want to avoid detecting drifts due to these removed traces, we only include the train dataset.


In [ ]:
log_path = data_path("real", "CALL", "CallCenter_train.csv.gz")
df = pd.read_csv(log_path, delimiter=",")

df["case:concept:name"] = df["case_id"]
df["concept:name"] = df["activity"]
df["time:timestamp"] = df["end_time"]
df["org:resource"] = df["resource"]

df = dataframe_utils.convert_timestamp_columns_in_df(df)
df = df.sort_values(by="time:timestamp").reset_index(drop=True)

event_log = log_converter.apply(df, variant=log_converter.Variants.TO_EVENT_LOG)
output_path = xes_out("CALL", "CALL")
xes_exporter.apply(event_log, str(output_path), parameters={"gzip": True})
print("Wrote", output_path, "| events:", len(df), "| traces:", df["case:concept:name"].nunique())


## ITHD

In [ ]:
log_path = data_path("real", "ITHD", "finale.csv.gz")
df = pd.read_csv(log_path, delimiter=",")

df["case:concept:name"] = df["Case ID"]
df["concept:name"] = df["Activity"]
df["time:timestamp"] = df["Complete Timestamp"]
df["org:resource"] = df["Resource"]

df = dataframe_utils.convert_timestamp_columns_in_df(df)
df = df.sort_values(by="time:timestamp").reset_index(drop=True)

event_log = log_converter.apply(df, variant=log_converter.Variants.TO_EVENT_LOG)
output_path = xes_out("ITHD", "ITHD")
xes_exporter.apply(event_log, str(output_path), parameters={"gzip": True})
print("Wrote", output_path, "| events:", len(df), "| traces:", df["case:concept:name"].nunique())


## MOBIS

In [ ]:
log_path = data_path("real", "MOBIS", "10258_mobis_challenge_log_2019.csv.gz")
df = pd.read_csv(log_path, delimiter=";")

df["case:concept:name"] = df["case"]
df["concept:name"] = df["activity"]
df["time:timestamp"] = df["end"]

df["time:timestamp"] = pd.to_datetime(
    df["time:timestamp"],
    format="%d.%m.%Y %H:%M",
    utc=True,
)
df = df.sort_values(by="time:timestamp").reset_index(drop=True)

event_log = log_converter.apply(df, variant=log_converter.Variants.TO_EVENT_LOG)
output_path = xes_out("MOBIS", "MOBIS")
xes_exporter.apply(event_log, str(output_path), parameters={"gzip": True})
print("Wrote", output_path, "| events:", len(df), "| traces:", df["case:concept:name"].nunique())


## CVS Pharmacy

In [ ]:
zip_path = data_path("synthetic", "cvs", "cvs_pharmacy.zip")
member = "cvs_pharmacy.xes"
output_path = data_path("synthetic", "cvs", "cvs_pharmacy.xes.gz")

with zipfile.ZipFile(zip_path) as zf, zf.open(member) as src, open(output_path, "wb") as out:
    with gzip.GzipFile(fileobj=out, mode="wb", filename=member) as gz:
        while True:
            chunk = src.read(1 << 20)
            if not chunk:
                break
            gz.write(chunk)

print("Wrote", output_path)